# Unbounded-alpha, lambda=1 abundant-data ablation

Thin driver cloned from the successful `alpha_u_ab` experiment. The only scientific change is `lambda_mix: 0.70 -> 1.00`; the unbounded alpha identifier and all benchmark/training settings are retained. Worker shards continue to skip the global merge cell; the launcher runs it once after all shards finish.


In [ ]:
from __future__ import annotations

import hashlib
import inspect
import json
import os
import re
from pathlib import Path

HERE = Path.cwd()
BASE_NOTEBOOK = HERE / "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed.ipynb"
if not BASE_NOTEBOOK.exists():
    raise FileNotFoundError(f"Missing committed abundant-data base notebook: {BASE_NOTEBOOK}")

print("Alpha-u abundant base:", BASE_NOTEBOOK.name)

base_nb = json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))
patched_identifier = False
patched_study = False
patched_results = False
patched_config = False
patched_lambda = False
lambda_patch_count = 0
patched_cells = []

for cell_index, cell in enumerate(base_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if "from opinion_dynamics.identify_nonlinear import (" in src:
        src = src.replace(
            "from opinion_dynamics.identify_nonlinear import (",
            "from opinion_dynamics.identify_nonlinear_unbounded import (",
            1,
        )
        patched_identifier = True

    src2, n = re.subn(
        r'STUDY_NAME\s*=\s*"[^"]+"',
        'STUDY_NAME = "lambda1_ab"',
        src,
        count=1,
    )
    if n:
        src = src2
        patched_study = True

    src, _ = re.subn(
        r'PIPELINE_VERSION\s*=\s*"[^"]+"',
        'PIPELINE_VERSION = "2026-09-14-lambda1-ab-v1"',
        src,
        count=1,
    )

    old_name = "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed"
    if old_name in src:
        src = src.replace(old_name, "lambda1_ab")
        patched_results = True

    if "SCIENTIFIC_CONFIG = {" in src and '"alpha_parameterization"' not in src:
        src = src.replace(
            "SCIENTIFIC_CONFIG = {\n",
            'SCIENTIFIC_CONFIG = {\n'
            '    "alpha_parameterization": "softplus_over_log2_positive_unbounded",\n',
            1,
        )
        patched_config = True


    # Lambda ablation: only change the control blend from 0.70 to 1.00.
    lambda_patterns = [
        r'(?i)(\blambda_mix\s*=\s*)0\.70\b',
        r'(?i)(\blambda_mix\s*=\s*)0\.7\b',
        r'(?i)(["\']lambda_mix["\']\s*:\s*)0\.70\b',
        r'(?i)(["\']lambda_mix["\']\s*:\s*)0\.7\b',
    ]
    for pat in lambda_patterns:
        src, n_lambda = re.subn(pat, r'\g<1>1.0', src)
        if n_lambda:
            lambda_patch_count += n_lambda
            patched_lambda = True

    if "SCIENTIFIC_CONFIG = {" in src and '"lambda_mix"' not in src and "'lambda_mix'" not in src:
        src = src.replace(
            "SCIENTIFIC_CONFIG = {\n",
            'SCIENTIFIC_CONFIG = {\n    "lambda_mix": 1.0,\n',
            1,
        )
        patched_lambda = True

    patched_cells.append((cell_index, src))

required = {
    "unbounded identifier import": patched_identifier,
    "short STUDY_NAME": patched_study,
    "short results namespace": patched_results,
    "alpha config marker": patched_config,
    "lambda_mix=1.0": patched_lambda,
}
missing = [k for k,v in required.items() if not v]
if missing:
    raise RuntimeError("Expected abundant benchmark structure missing: " + ", ".join(missing))

print("Alpha-u abundant + lambda=1 overrides validated.")
print("Lambda numeric replacements:", lambda_patch_count)
worker_mode = os.environ.get("ABUNDANT_FULL_SHARD_ID") not in (None, "")
print("Driver mode:", "worker" if worker_mode else "final validation/merge")

cells_to_run = patched_cells[:-1] if worker_mode else patched_cells
if worker_mode:
    print("Worker mode: skipping base notebook final global validation/merge cell.")

g = globals()
for cell_index, src in cells_to_run:
    print(f"[base code cell {cell_index}]")
    exec(compile(src, f"{BASE_NOTEBOOK.name}:cell_{cell_index}", "exec"), g, g)
